# DeepSeek: Strategy Usage Over Time and Transitions

Reads annotated+parsed traces (`*_annot_parsed.csv`, produced by `040-annotation.ipynb`) and produces DeepSeek's figures: the main-text temporal + transition figure (Figure 2, `figures/deepseek_reasoning_grid_varB.pdf`, main text Table 3's companion figure), the appendix CoT counterparts (Figure 3, Figure 6, Appendix B.3 / Appendix B.4), and DeepSeek's rows of the occurrence table (Table 18, Appendix B.2). GLM analog is `052-process-glm.ipynb` (no main-figure grid there -- GLM only needs the appendix figures).

In [ ]:
import re
import pandas as pd
import glob
import ast
import numpy as np
import matplotlib.pyplot as plt
import os
from collections import Counter
import scipy.stats as st

REASONING_STEM = "deepseek-naive-deepseek-reasoner"
COT_STEM = "deepseek-naive-cot-deepseek-chat"

# Set up output directory
OUTPUT_DIR = "figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define the annotation label pattern and keywords grouped by topic
tag_labels = [
    "<INTER>",
    "<CORR>",
    "<ERR_DESC>",
    "<INST>",
    "<ERR_SIM>",
    "<PLAUS>",
    "<CURATE>",
    "<RECON>",
    "<LINK>",
    "<DISCR>",
]

# Extract component names
tag_names = [lbl.strip("<>").lower() for lbl in tag_labels]

# Define color palette
# Paul Tol's "muted" palette - colorblind-safe, grayscale-safe, designed for scientific publication
colors = [
    "#88CCEE",  # cyan       -> interpretation
    "#117733",  # green      -> correct answer ref
    "#CC6677",  # rose       -> error description
    "#DDCC77",  # sand       -> outcome instantiation
    "#AA4499",  # purple     -> error simulation
    "#332288",  # indigo     -> plausibility check
    "#44AA99",  # teal       -> final set curation
    "#000000",  # black      -> reconsideration
    "#999933",  # olive      -> link
    "#882255",  # wine       -> discriminator
]
color_map = {tag: colors[i] for i, tag in enumerate(tag_names)}

# Order to use for legends (matches taxonomy table: Understanding / Generation / Evaluation / Selection-Reflection)
LEGEND_ORDER = ["inter", "corr", "link", "err_desc", "err_sim", "inst", "plaus", "discr", "curate", "recon"]

TAG_DISPLAY_NAMES = {
    "inter": "Task Interpretation",
    "corr": "Correct Answer Ref.",
    "err_desc": "Error Description",
    "inst": "Outcome Instantiation",
    "err_sim": "Error Simulation",
    "plaus": "Plausibility Check",
    "curate": "Final Set Curation",
    "recon": "Reconsideration",
    "link": "Conceptual Link",
    "discr": "Discriminability Check",
}

# Set up styling
BASE_FONT_SIZE = 13
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times", "STIXGeneral", "TeX Gyre Termes"],
    "font.size": BASE_FONT_SIZE * 1.5,
    "axes.titlesize": BASE_FONT_SIZE * 1.5,
    "axes.labelsize": BASE_FONT_SIZE * 1.5,
    "xtick.labelsize": BASE_FONT_SIZE * 1.5,
    "ytick.labelsize": BASE_FONT_SIZE * 1.5,
    "legend.fontsize": BASE_FONT_SIZE * 1.2,
    "figure.titlesize": BASE_FONT_SIZE * 1.5,
})


In [ ]:
# Load and Parse Annotated Data

def load_and_parse_traces(pattern):
    """
    Load annotated traces from CSV files matching the pattern.
    Returns a dictionary with tag names as keys and lists of normalized positions as values.
    """
    positions_by_tag = {tag: [] for tag in tag_names}
    parsed_csv_paths = glob.glob(pattern)
    for path in parsed_csv_paths:
        df = pd.read_csv(path)
        for _, row in df.iterrows():
            reasoning = row.get("trace", "")
            seq_str = row.get("annotation_sequence", "")
            seq = ast.literal_eval(seq_str) if isinstance(seq_str, str) and seq_str.startswith("[") else []
            reasoning_len = len(reasoning) if reasoning else 1
            for pos, tag in seq:
                if tag in positions_by_tag:
                    positions_by_tag[tag].append(pos / reasoning_len)
    return positions_by_tag


def patterns_for(dataset_dir, regime):
    """regime in {'reasoning', 'cot'}"""
    stem = REASONING_STEM if regime == "reasoning" else COT_STEM
    return f"{dataset_dir}/joint_results/annotated/{stem}_*_annot_parsed.csv"


print("Loading EEDI reasoning traces...")
eedi_reasoning_positions = load_and_parse_traces(patterns_for("eedi_data", "reasoning"))
print(f"  {sum(len(v) for v in eedi_reasoning_positions.values())} occurrences")

print("Loading EEDI CoT traces...")
eedi_cot_positions = load_and_parse_traces(patterns_for("eedi_data", "cot"))
print(f"  {sum(len(v) for v in eedi_cot_positions.values())} occurrences")

print("Loading SciQ reasoning traces...")
sciq_reasoning_positions = load_and_parse_traces(patterns_for("sciq_data", "reasoning"))
print(f"  {sum(len(v) for v in sciq_reasoning_positions.values())} occurrences")

print("Loading SciQ CoT traces...")
sciq_cot_positions = load_and_parse_traces(patterns_for("sciq_data", "cot"))
print(f"  {sum(len(v) for v in sciq_cot_positions.values())} occurrences")


In [ ]:
# ============ CONFIGURATION ============
SHOW_ERROR_BARS = False  # Toggle error bars here
bins = np.linspace(0, 1, 6)
# ======================================

def get_mean_and_ci(scores: np.ndarray, confidence: float = 0.95) -> tuple[float, float]:
    """Mean and confidence interval using t-distribution."""
    mean = np.mean(scores)
    sem = st.sem(scores)
    n = len(scores)
    h = sem * st.t.ppf((1 + confidence) / 2, n - 1)
    return mean, h


def compute_line_data(positions_by_tag, bins=np.linspace(0, 1, 6)):
    bin_centers = (bins[:-1] + bins[1:]) / 2
    line_data = {}
    line_ci = {}
    for tag in tag_names:
        data = np.array(positions_by_tag.get(tag, []))
        bin_counts, _ = np.histogram(data, bins=bins)
        line_data[tag] = bin_counts
        bin_cis = []
        for bin_idx in range(len(bins) - 1):
            bin_mask = (data >= bins[bin_idx]) & (data < bins[bin_idx + 1])
            bin_positions = data[bin_mask]
            if len(bin_positions) > 1:
                _, ci = get_mean_and_ci(bin_positions)
                bin_cis.append(ci)
            else:
                bin_cis.append(0)
        line_ci[tag] = np.array(bin_cis)
    return line_data, line_ci, bin_centers


def normalize_by_timebucket(line_data):
    normalized = {}
    n_bins = len(next(iter(line_data.values())))
    for tag in tag_names:
        normalized[tag] = np.zeros(n_bins)
    for bin_idx in range(n_bins):
        bin_sum = sum(line_data[tag][bin_idx] for tag in tag_names)
        if bin_sum > 0:
            for tag in tag_names:
                normalized[tag][bin_idx] = line_data[tag][bin_idx] / bin_sum
    return normalized


def normalize_ci_by_timebucket(line_data, line_ci):
    normalized_ci = {}
    n_bins = len(next(iter(line_data.values())))
    for tag in tag_names:
        normalized_ci[tag] = np.zeros(n_bins)
    for bin_idx in range(n_bins):
        bin_sum = sum(line_data[tag][bin_idx] for tag in tag_names)
        if bin_sum > 0:
            for tag in tag_names:
                normalized_ci[tag][bin_idx] = line_ci[tag][bin_idx] / bin_sum
    return normalized_ci


def plot_dataset_split(eedi_positions, sciq_positions, regime_title):
    """Build a figure with EEDI on the left and SciQ on the right for one regime."""
    eedi_ld, eedi_ci, bin_centers = compute_line_data(eedi_positions, bins)
    sciq_ld, sciq_ci, _ = compute_line_data(sciq_positions, bins)

    eedi_norm = normalize_by_timebucket(eedi_ld)
    sciq_norm = normalize_by_timebucket(sciq_ld)
    eedi_norm_ci = normalize_ci_by_timebucket(eedi_ld, eedi_ci) if SHOW_ERROR_BARS else None
    sciq_norm_ci = normalize_ci_by_timebucket(sciq_ld, sciq_ci) if SHOW_ERROR_BARS else None

    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

    for ax, norm, norm_ci, title, show_ylabel in [
        (ax_left, eedi_norm, eedi_norm_ci, "Eedi", True),
        (ax_right, sciq_norm, sciq_norm_ci, "SciQ", False),
    ]:
        for tag in tag_names:
            if SHOW_ERROR_BARS and norm_ci is not None:
                ax.errorbar(bin_centers, norm[tag], yerr=norm_ci[tag],
                            marker='o', linewidth=2, markersize=8, capsize=4,
                            label=TAG_DISPLAY_NAMES.get(tag, tag), color=color_map[tag])
            else:
                ax.plot(bin_centers, norm[tag], marker='o', linewidth=2, markersize=8,
                        label=TAG_DISPLAY_NAMES.get(tag, tag), color=color_map[tag])
        ax.set_xlabel('Normalized Position In Trace', fontsize=BASE_FONT_SIZE*1.5, fontweight='bold')
        ax.set_title(title, fontsize=BASE_FONT_SIZE*1.5, fontweight='bold')
        ax.set_xlim(bins[0], bins[-1])
        ax.grid(True, alpha=0.3)
        if show_ylabel:
            ax.set_ylabel('Share of Strategies', fontsize=BASE_FONT_SIZE*1.5, fontweight='bold')

    fig.legend(
        handles=[plt.Line2D([0], [0], color=color_map[tag], marker='o', lw=2, markersize=8,
                            label=TAG_DISPLAY_NAMES.get(tag, tag)) for tag in LEGEND_ORDER],
        loc='center left',
        bbox_to_anchor=(0.97, 0.5),
        ncol=1,
        fontsize=BASE_FONT_SIZE*1.5,
        frameon=True,
        framealpha=0.95,
        labelspacing=0.25,
        borderpad=0.5,
        handlelength=1.5,
    )

    plt.tight_layout(w_pad=1.0)
    return fig


print("Building Chain-of-Thought figure...")
fig_cot = plot_dataset_split(eedi_cot_positions, sciq_cot_positions, "Chain-of-Thought")
plt.show()

print("Building Reasoning figure...")
fig_reasoning = plot_dataset_split(eedi_reasoning_positions, sciq_reasoning_positions, "Reasoning Traces")
plt.show()


In [ ]:
# Export Figures

out_pdf_cot = os.path.join(OUTPUT_DIR, "deepseek_cot_components_over_time.pdf")
fig_cot.savefig(out_pdf_cot, bbox_inches="tight", dpi=300)
print(f"Saved CoT PDF: {out_pdf_cot}")
# fig_cot.savefig(os.path.join(OUTPUT_DIR, "deepseek_cot_components_over_time.png"), bbox_inches="tight", dpi=300)

out_pdf_reasoning = os.path.join(OUTPUT_DIR, "deepseek_reasoning_components_over_time.pdf")
fig_reasoning.savefig(out_pdf_reasoning, bbox_inches="tight", dpi=300)
print(f"Saved Reasoning PDF: {out_pdf_reasoning}")
# fig_reasoning.savefig(os.path.join(OUTPUT_DIR, "deepseek_reasoning_components_over_time.png"), bbox_inches="tight", dpi=300)


In [ ]:
# Summary Statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)


def compute_trace_stats(pattern):
    paths = glob.glob(pattern)
    total_traces = 0
    total_len = 0
    presence = {tag: 0 for tag in tag_names}
    counts_per_trace = {tag: [] for tag in tag_names}
    for path in paths:
        df = pd.read_csv(path)
        for _, row in df.iterrows():
            trace = row.get("trace", "")
            seq_str = row.get("annotation_sequence", "")
            seq = ast.literal_eval(seq_str) if isinstance(seq_str, str) and seq_str.startswith("[") else []
            total_traces += 1
            total_len += len(trace) if isinstance(trace, str) else 0
            labels_list = [label for _, label in seq if label in tag_names]
            for tag in set(labels_list):
                presence[tag] += 1
            counts = Counter(labels_list)
            for tag in tag_names:
                counts_per_trace[tag].append(counts.get(tag, 0))
    avg_len = (total_len / total_traces) if total_traces > 0 else 0
    presence_pct = {tag: (presence[tag] / total_traces * 100) if total_traces > 0 else 0 for tag in tag_names}
    return total_traces, avg_len, presence_pct, counts_per_trace


def report(label, pattern):
    n, avg_len, pres_pct, counts = compute_trace_stats(pattern)
    print(f"\n[{label}] n={n}, avg length={avg_len:.1f} chars")
    print("  Avg occurrences per trace (mean +/- 95% CI):")
    for tag in tag_names:
        arr = np.array(counts[tag])
        if len(arr) > 1:
            mean, ci = get_mean_and_ci(arr)
        else:
            mean, ci = (np.mean(arr) if len(arr) > 0 else 0), 0
        print(f"    {TAG_DISPLAY_NAMES.get(tag, tag)}: {mean:.2f} +/- {ci:.2f}")
    print("  Presence (% of traces with >=1 occurrence):")
    for tag in tag_names:
        print(f"    {TAG_DISPLAY_NAMES.get(tag, tag)}: {pres_pct[tag]:.1f}%")


for label, regime, dataset in [
    ("EEDI / Reasoning", "reasoning", "eedi_data"),
    ("EEDI / CoT",        "cot",       "eedi_data"),
    ("SciQ / Reasoning", "reasoning", "sciq_data"),
    ("SciQ / CoT",        "cot",       "sciq_data"),
]:
    report(label, patterns_for(dataset, regime))


In [ ]:
# ---------- strategy-transition Sankey (both regimes) ----------
# CoT output cited by Figure 6 (Appendix B.4).
# Reasoning output isn't separately cited (DeepSeek reasoning's transition figure is the main-text grid below),
# generated anyway for parity with 052-process-glm.ipynb.
import plotly.graph_objects as go
from collections import Counter, defaultdict


def _to_rgb_tuple(color):
    if isinstance(color, (list, tuple)):
        if max(color) <= 1.0:
            return tuple(int(c*255) for c in color)
        return tuple(int(c) for c in color)
    color = str(color).strip()
    if color.startswith('#'):
        h = color.lstrip('#')
        return (int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16))
    m = re.match(r'rgb\((\d+),(\d+),(\d+)\)', color)
    if m:
        return (int(m.group(1)), int(m.group(2)), int(m.group(3)))
    return (0, 0, 0)


def plot_sankey_tag_transitions(
    parsed_csv_paths,
    tag_labels,
    title,
    output_path,
    show_legend,
    nr_sankey_steps=3,
    min_transition_count=1,
    merge_consecutive: set = None,
    min_outgoing_mass=0.1,
    max_nr_outgoing_edges=None,
):
    merge_consecutive = merge_consecutive or set()
    tag_names_local = [lbl.strip("<>").lower() for lbl in tag_labels if lbl != "unknown"]

    FONT_FAMILY = "Times New Roman, Times, STIXGeneral, TeX Gyre Termes, serif"
    BASE_FONT_SIZE_S = 15
    TITLE_FONT_SIZE = int(BASE_FONT_SIZE_S * 2)
    LEGEND_FONT_SIZE = int(BASE_FONT_SIZE_S * 2)

    all_sequences = []
    for path in parsed_csv_paths:
        df = pd.read_csv(path)
        for seq_str in df["annotation_sequence"].fillna(""):
            seq = ast.literal_eval(seq_str) if isinstance(seq_str, str) and seq_str.startswith("[") else []
            labels = [label for _, label in seq if label in tag_names_local]
            if not labels:
                continue
            labels = [l1 for l1, l2 in zip(labels[:-1], labels[1:]) if ((l1 != l2) or (l1 not in merge_consecutive))] + [labels[-1]]
            if len(labels) >= nr_sankey_steps:
                for i in range(len(labels) - nr_sankey_steps + 1):
                    all_sequences.append(labels[i:i+nr_sankey_steps])

    transitions = defaultdict(Counter)
    for seq in all_sequences:
        for step in range(nr_sankey_steps-1):
            transitions[(step, seq[step])][seq[step+1]] += 1

    node_labels = []
    node_indices = {}
    node_colors = []
    idx = 0
    for step in range(nr_sankey_steps):
        for tag in tag_names_local:
            node_labels.append("")
            node_indices[(step, tag)] = idx
            node_colors.append(color_map[tag])
            idx += 1

    total_outgoing = defaultdict(int)
    for step in range(nr_sankey_steps-1):
        for from_tag in tag_names_local:
            for to_tag, count in transitions[(step, from_tag)].items():
                if count >= min_transition_count:
                    total_outgoing[(step, from_tag)] += count

    top_edges = set()
    if max_nr_outgoing_edges is not None:
        for step in range(nr_sankey_steps-1):
            for from_tag in tag_names_local:
                outgoing = sorted(transitions[(step, from_tag)].items(), key=lambda x: x[1], reverse=True)
                for to_tag, count in outgoing[:max_nr_outgoing_edges]:
                    if count >= min_transition_count:
                        top_edges.add((step, from_tag, to_tag))

    sources, targets, values, link_colors = [], [], [], []
    for step in range(nr_sankey_steps-1):
        for from_tag in tag_names_local:
            total_mass = total_outgoing[(step, from_tag)]
            for to_tag, count in transitions[(step, from_tag)].items():
                if count >= min_transition_count:
                    sources.append(node_indices[(step, from_tag)])
                    targets.append(node_indices[(step+1, to_tag)])
                    values.append(count)
                    is_transparent = False
                    if total_mass > 0 and count / total_mass < min_outgoing_mass:
                        is_transparent = True
                    if max_nr_outgoing_edges is not None and (step, from_tag, to_tag) not in top_edges:
                        is_transparent = True
                    if is_transparent:
                        link_colors.append("rgba(0,0,0,0)")
                    else:
                        link_colors.append(color_map[from_tag])

    fig = go.Figure(data=[go.Sankey(
        node=dict(pad=5, thickness=15, line=dict(color="black", width=0.5),
                  label=node_labels, color=node_colors),
        link=dict(source=sources, target=targets, value=values, color=link_colors),
    )])

    for tag in [t for t in LEGEND_ORDER if t in tag_names_local]:
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='markers',
            marker=dict(size=10, color=color_map[tag]),
            legendgroup=tag,
            showlegend=show_legend,
            name=TAG_DISPLAY_NAMES.get(tag, tag),
        ))

    layout_kwargs = dict(
        title_text=title,
        title_x=0.33 if show_legend else 0.5,
        title_xanchor="center",
        title_y=0.98,
        title_yanchor="top",
        margin=dict(t=40, b=60, l=40, r=40),
        title_font=dict(family=FONT_FAMILY, size=TITLE_FONT_SIZE, color="black", weight="bold"),
        font=dict(family=FONT_FAMILY, size=BASE_FONT_SIZE_S),
        legend_font=dict(family=FONT_FAMILY, size=LEGEND_FONT_SIZE, color="black"),
        xaxis=dict(title="Step",
                   title_font=dict(size=TITLE_FONT_SIZE, family=FONT_FAMILY, color="black", weight="bold"),
                   showticklabels=False, showgrid=False, zeroline=False, showline=False),
        yaxis=dict(title="Strategy" if not show_legend else "",
                   title_font=dict(size=TITLE_FONT_SIZE, family=FONT_FAMILY, color="black", weight="bold"),
                   visible=show_legend is False or True,
                   showticklabels=False, showgrid=False, zeroline=False, showline=False),
        height=450,
        width=800 if show_legend else 500,
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    if show_legend:
        layout_kwargs["legend"] = dict(
            traceorder="normal", itemclick="toggle", itemsizing="constant",
            yanchor="top", y=1.0, xanchor="right", x=1.75,
            font=dict(family=FONT_FAMILY, size=int(BASE_FONT_SIZE_S * 1.6), color="black"),
            tracegroupgap=1,
        )
    fig.update_layout(**layout_kwargs)
    fig.write_image(output_path, width=layout_kwargs["width"], height=450)
    fig.show()


nr_sankey_steps = 4
min_outgoing_mass = 0.15

for stem, regime in [(COT_STEM, "cot"), (REASONING_STEM, "reasoning")]:
    eedi_paths = glob.glob(f"eedi_data/joint_results/annotated/{stem}_*_annot_parsed.csv")
    sciq_paths = glob.glob(f"sciq_data/joint_results/annotated/{stem}_*_annot_parsed.csv")

    # EEDI on the left -> no legend (SciQ panel on the right carries the shared legend)
    plot_sankey_tag_transitions(
        eedi_paths, tag_labels,
        title="Eedi",
        output_path=f"figures/deepseek_{regime}_eedi_subprocesses.pdf",
        show_legend=False,
        nr_sankey_steps=nr_sankey_steps,
        min_transition_count=1,
        min_outgoing_mass=min_outgoing_mass,
        max_nr_outgoing_edges=1000,
    )

    # SciQ on the right -> with legend
    plot_sankey_tag_transitions(
        sciq_paths, tag_labels,
        title="SciQ",
        output_path=f"figures/deepseek_{regime}_sciq_subprocesses.pdf",
        show_legend=True,
        nr_sankey_steps=nr_sankey_steps,
        min_transition_count=1,
        min_outgoing_mass=min_outgoing_mass,
        max_nr_outgoing_edges=1000,
    )


In [ ]:
# Shared helper for variants: builds transitions, then lets caller customize style.

def _build_sankey_data(parsed_csv_paths, tag_labels, nr_sankey_steps=4,
                       min_transition_count=1, min_outgoing_mass=0.15,
                       max_nr_outgoing_edges=None, merge_consecutive=None):
    merge_consecutive = merge_consecutive or set()
    tag_names_local = [lbl.strip("<>").lower() for lbl in tag_labels if lbl != "unknown"]

    all_sequences = []
    for path in parsed_csv_paths:
        df = pd.read_csv(path)
        for seq_str in df["annotation_sequence"].fillna(""):
            seq = ast.literal_eval(seq_str) if isinstance(seq_str, str) and seq_str.startswith("[") else []
            labels = [label for _, label in seq if label in tag_names_local]
            if not labels:
                continue
            labels = [l1 for l1, l2 in zip(labels[:-1], labels[1:])
                      if ((l1 != l2) or (l1 not in merge_consecutive))] + [labels[-1]]
            if len(labels) >= nr_sankey_steps:
                for i in range(len(labels) - nr_sankey_steps + 1):
                    all_sequences.append(labels[i:i + nr_sankey_steps])

    transitions = defaultdict(Counter)
    for seq in all_sequences:
        for step in range(nr_sankey_steps - 1):
            transitions[(step, seq[step])][seq[step + 1]] += 1

    node_indices = {}
    node_colors = []
    idx = 0
    for step in range(nr_sankey_steps):
        for tag in tag_names_local:
            node_indices[(step, tag)] = idx
            node_colors.append(color_map[tag])
            idx += 1

    total_outgoing = defaultdict(int)
    for step in range(nr_sankey_steps - 1):
        for from_tag in tag_names_local:
            for to_tag, count in transitions[(step, from_tag)].items():
                if count >= min_transition_count:
                    total_outgoing[(step, from_tag)] += count

    top_edges = set()
    if max_nr_outgoing_edges is not None:
        for step in range(nr_sankey_steps - 1):
            for from_tag in tag_names_local:
                outgoing = sorted(transitions[(step, from_tag)].items(),
                                  key=lambda x: x[1], reverse=True)
                for to_tag, count in outgoing[:max_nr_outgoing_edges]:
                    if count >= min_transition_count:
                        top_edges.add((step, from_tag, to_tag))

    edges = []  # list of (source_idx, target_idx, value, visible, from_tag)
    for step in range(nr_sankey_steps - 1):
        for from_tag in tag_names_local:
            total_mass = total_outgoing[(step, from_tag)]
            for to_tag, count in transitions[(step, from_tag)].items():
                if count < min_transition_count:
                    continue
                visible = True
                if total_mass > 0 and count / total_mass < min_outgoing_mass:
                    visible = False
                if max_nr_outgoing_edges is not None and (step, from_tag, to_tag) not in top_edges:
                    visible = False
                edges.append((
                    node_indices[(step, from_tag)],
                    node_indices[(step + 1, to_tag)],
                    count, visible, from_tag,
                ))

    n_nodes = len(node_colors)
    return tag_names_local, node_colors, edges, n_nodes


def _render_sankey(node_colors, edges, link_colors, title, output_path,
                   show_legend, node_thickness=15, node_pad=5,
                   tag_names_local=None):
    FONT_FAMILY = "Times New Roman, Times, STIXGeneral, TeX Gyre Termes, serif"
    BASE_FONT_SIZE_S = 15
    TITLE_FONT_SIZE = int(BASE_FONT_SIZE_S * 2)
    LEGEND_FONT_SIZE = int(BASE_FONT_SIZE_S * 2)

    sources = [e[0] for e in edges]
    targets = [e[1] for e in edges]
    values = [e[2] for e in edges]

    fig = go.Figure(data=[go.Sankey(
        node=dict(pad=node_pad, thickness=node_thickness,
                  line=dict(color="black", width=0.5),
                  label=["" for _ in node_colors], color=node_colors),
        link=dict(source=sources, target=targets, value=values, color=link_colors),
    )])

    if tag_names_local is None:
        tag_names_local = [lbl.strip("<>").lower() for lbl in tag_labels if lbl != "unknown"]
    for tag in [t for t in LEGEND_ORDER if t in tag_names_local]:
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='markers',
            marker=dict(size=10, color=color_map[tag]),
            legendgroup=tag,
            showlegend=show_legend,
            name=TAG_DISPLAY_NAMES.get(tag, tag),
        ))

    layout_kwargs = dict(
        title_text=title,
        title_x=0.33 if show_legend else 0.5,
        title_xanchor="center", title_y=0.98, title_yanchor="top",
        margin=dict(t=40, b=60, l=40, r=40),
        title_font=dict(family=FONT_FAMILY, size=TITLE_FONT_SIZE, color="black", weight="bold"),
        font=dict(family=FONT_FAMILY, size=BASE_FONT_SIZE_S),
        legend_font=dict(family=FONT_FAMILY, size=LEGEND_FONT_SIZE, color="black"),
        xaxis=dict(title="Step",
                   title_font=dict(size=TITLE_FONT_SIZE, family=FONT_FAMILY, color="black", weight="bold"),
                   showticklabels=False, showgrid=False, zeroline=False, showline=False),
        yaxis=dict(title="Strategy" if not show_legend else "",
                   title_font=dict(size=TITLE_FONT_SIZE, family=FONT_FAMILY, color="black", weight="bold"),
                   showticklabels=False, showgrid=False, zeroline=False, showline=False),
        height=450,
        width=800 if show_legend else 500,
        paper_bgcolor="white", plot_bgcolor="white",
    )
    if show_legend:
        layout_kwargs["legend"] = dict(
            traceorder="normal", itemclick="toggle", itemsizing="constant",
            yanchor="top", y=1.0, xanchor="right", x=1.75,
            font=dict(family=FONT_FAMILY, size=int(BASE_FONT_SIZE_S * 1.6), color="black"),
            tracegroupgap=1,
        )
    fig.update_layout(**layout_kwargs)
    fig.write_image(output_path, width=layout_kwargs["width"], height=450)
    fig.show()

## Main-text figure: strategy usage over time + transitions (`deepseek_reasoning_grid_varB.pdf`)

Combined 2x2 grid (Eedi/SciQ x line-plot/Sankey) for the reasoning setting -- this is Figure 2 in the main paper. Node thickness is widened ("varB") over earlier plain-Sankey attempts for legibility.

In [ ]:
import io
import matplotlib.image as mpimg


def render_grid(regime, sankey_kwargs, suffix):
    """2x2 grid: top row = strategy-share-over-time lines (Eedi, SciQ), bottom row =
    strategy-transition Sankeys (Eedi, SciQ), for one prompting regime. Saves to
    figures/deepseek_{regime}_grid_{suffix}.pdf.
    """
    stem = REASONING_STEM if regime == "reasoning" else COT_STEM
    eedi_positions = eedi_reasoning_positions if regime == "reasoning" else eedi_cot_positions
    sciq_positions = sciq_reasoning_positions if regime == "reasoning" else sciq_cot_positions

    eedi_ld, _, bin_centers = compute_line_data(eedi_positions, bins)
    sciq_ld, _, _ = compute_line_data(sciq_positions, bins)
    eedi_norm = normalize_by_timebucket(eedi_ld)
    sciq_norm = normalize_by_timebucket(sciq_ld)

    eedi_paths = glob.glob(f"eedi_data/joint_results/annotated/{stem}_*_annot_parsed.csv")
    sciq_paths = glob.glob(f"sciq_data/joint_results/annotated/{stem}_*_annot_parsed.csv")

    sankey_build_kwargs = dict(
        nr_sankey_steps=sankey_kwargs["nr_sankey_steps"],
        min_transition_count=sankey_kwargs["min_transition_count"],
        min_outgoing_mass=sankey_kwargs["min_outgoing_mass"],
        max_nr_outgoing_edges=sankey_kwargs["max_nr_outgoing_edges"],
    )

    def _sankey_png(paths, title, show_legend):
        tag_names_local, node_colors, edges, n_nodes = _build_sankey_data(paths, tag_labels, **sankey_build_kwargs)
        link_colors = [color_map[e[4]] if e[3] else "rgba(0,0,0,0)" for e in edges]
        sources = [e[0] for e in edges]
        targets = [e[1] for e in edges]
        values = [e[2] for e in edges]

        fig = go.Figure(data=[go.Sankey(
            node=dict(pad=sankey_kwargs["node_pad"], thickness=sankey_kwargs["node_thickness"],
                      line=dict(color="black", width=sankey_kwargs.get("node_line_width", 0.5)),
                      label=["" for _ in node_colors], color=node_colors),
            link=dict(source=sources, target=targets, value=values, color=link_colors),
        )])
        for tag in [t for t in LEGEND_ORDER if t in tag_names_local]:
            fig.add_trace(go.Scatter(
                x=[None], y=[None], mode="markers",
                marker=dict(size=10, color=color_map[tag]),
                legendgroup=tag, showlegend=show_legend,
                name=TAG_DISPLAY_NAMES.get(tag, tag),
            ))
        fig.update_layout(
            title_text=title, title_x=0.5, margin=dict(t=30, b=10, l=10, r=10),
            paper_bgcolor="white", plot_bgcolor="white",
            width=sankey_kwargs["width"], height=sankey_kwargs["height"],
            showlegend=show_legend,
            legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5) if show_legend else None,
        )
        png_bytes = fig.to_image(format="png", width=sankey_kwargs["width"], height=sankey_kwargs["height"], scale=2)
        return mpimg.imread(io.BytesIO(png_bytes), format="png")

    eedi_sankey_img = _sankey_png(eedi_paths, "Eedi", show_legend=False)
    sciq_sankey_img = _sankey_png(sciq_paths, "SciQ", show_legend=True)

    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    for ax, norm, title, show_ylabel in [
        (axes[0, 0], eedi_norm, "Eedi", True),
        (axes[0, 1], sciq_norm, "SciQ", False),
    ]:
        for tag in tag_names:
            ax.plot(bin_centers, norm[tag], marker="o", linewidth=2, markersize=6,
                    label=TAG_DISPLAY_NAMES.get(tag, tag), color=color_map[tag])
        ax.set_xlabel("Normalized Position In Trace")
        ax.set_title(title, fontweight="bold")
        ax.set_xlim(bins[0], bins[-1])
        ax.grid(True, alpha=0.3)
        if show_ylabel:
            ax.set_ylabel("Share of Strategies")

    axes[0, 1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=BASE_FONT_SIZE)

    for ax, img in [(axes[1, 0], eedi_sankey_img), (axes[1, 1], sciq_sankey_img)]:
        ax.imshow(img)
        ax.axis("off")

    plt.tight_layout()
    out_path = f"figures/deepseek_{regime}_grid_{suffix}.pdf"
    fig.savefig(out_path, bbox_inches="tight", dpi=300)
    print(f"Saved {out_path}")
    plt.show()


variant_B_sankey_kwargs = dict(
    edge_style="by_source",
    node_thickness=40, node_pad=10,
    node_line_width=2,
    nr_sankey_steps=4,
    min_transition_count=1,
    min_outgoing_mass=0.15,
    max_nr_outgoing_edges=1000,
    width=900, height=500,
)

# -> figures/deepseek_reasoning_grid_varB.pdf, embedded as Figure 2.
render_grid("reasoning", variant_B_sankey_kwargs, suffix="varB")
